# 01 — Análisis Inicial de Calidad de los Datos

Este notebook realiza el perfilado inicial de calidad del dataset inmobiliario limpio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("data/processed/properties_clean.csv")
OUTPUTS = Path("outputs/figures")
OUTPUTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

df.head()

## 1. Dimensiones del dataset

In [ ]:
print("Número de registros:", df.shape[0])
print("Número de variables:", df.shape[1])

## 2. Tipos de datos

In [ ]:
df.dtypes

## 3. Valores nulos

In [ ]:
nulls_df = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().sum() / len(df) * 100).round(2)
})

nulls_df

## 4. Valores blancos

In [ ]:
text_columns = df.select_dtypes(include=["object"]).columns

blank_counts = {}

for col in text_columns:
    blank_counts[col] = df[col].astype(str).str.strip().eq("").sum()

blank_df = pd.DataFrame.from_dict(
    blank_counts,
    orient="index",
    columns=["valores_blancos"]
)

blank_df

## 5. Duplicados

In [ ]:
print("Duplicados exactos:", df.duplicated().sum())

print("Duplicados por source + property_id:")
print(df.duplicated(subset=["source", "property_id"]).sum())

## 6. Estadísticos descriptivos

In [ ]:
df.describe()

## 7. Cardinalidad de variables categóricas

In [ ]:
categorical_columns = ["source", "city", "country", "neighborhood", "property_type"]

for col in categorical_columns:
    print("\n---", col.upper(), "---")
    print("Valores únicos:", df[col].nunique())
    print(df[col].value_counts())

## 8. Detección de outliers mediante IQR

In [ ]:
numeric_columns = ["price_eur", "area_m2", "price_per_m2"]

for col in numeric_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)]

    print(f"\nVariable: {col}")
    print(f"Límite inferior: {lower_limit:.2f}")
    print(f"Límite superior: {upper_limit:.2f}")
    print(f"Outliers detectados: {len(outliers)}")

## 9. Heatmap de valores nulos

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False)
plt.title("Mapa de calor de valores nulos")
plt.savefig(OUTPUTS / "eda01_01_heatmap_nulos.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Histogramas iniciales

In [ ]:
for col in ["price_eur", "area_m2", "price_per_m2"]:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribución inicial de {col}")
    plt.savefig(OUTPUTS / f"eda01_hist_{col}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 11. Boxplot precio por m² por ciudad

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x="city", y="price_per_m2", data=df)
plt.title("Precio por m² por ciudad")
plt.savefig(OUTPUTS / "eda01_02_boxplot_precio_m2_ciudad.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. Interpretación inicial

El dataset limpio no presenta valores nulos ni duplicados en las claves principales. Las variables numéricas muestran dispersión elevada, especialmente en precio total y precio por metro cuadrado, algo coherente con el mercado inmobiliario premium. Los outliers detectados no deben eliminarse automáticamente porque pueden representar viviendas ultra luxury reales.